Goal:
take ip2location_2026-02-16.tar.gz, join with census of 2026-02-18

outcome -> assess usage types of anycast IPs in census

In [1]:
import pandas as pd
from pathlib import Path

import sys
analysis_dir = Path.cwd().parent
sys.path.append(str(analysis_dir))
from add_ASN import CaidaASLookup
import census_helper
from datetime import datetime

ts = datetime(2026, 2, 18)

import matplotlib.pyplot as plt

# configure matplotlib variables (font and size)
plt.rcParams["axes.labelsize"] = 20
plt.rcParams.update({'font.size': 14})


In [2]:
census_v6 = census_helper.download_date(ts, 'v6')
# filter on GCD-confirmed (high-confidence)
census_v6 = census_v6[census_v6['GCD_ICMPv6'] > 1]
# filter on relevant columns
census_v6 = census_v6[['prefix', 'ASN']]

census_v6['prefix'].nunique()

13007

In [3]:
census_v4 = census_helper.download_date(ts, 'v4')
# filter on GCD-confirmed (high-confidence)
census_v4 = census_v4[census_v4['GCD_ICMPv4'] > 1]
# filter on relevant columns
census_v4 = census_v4[['prefix', 'ASN']]

census_v4.head()

,prefix,ASN
0,1.0.0.0/24,13335
1,1.1.1.0/24,13335
3,1.10.10.0/24,148000
4,1.12.0.0/24,132203_45090
5,1.12.12.0/24,132203_45090


In [4]:
census_v4['prefix'].nunique()

14346

In [5]:
import ipaddress
import numpy as np

# Convert prefix strings to base IPv4 integers (network address)
base_ints = np.array([
    int(ipaddress.IPv4Address(p.split('/')[0]))
    for p in census_v4['prefix']
])

# For each /24, generate all 256 IPs (offsets 0-255)
all_ip_ints = (base_ints[:, np.newaxis] + np.arange(256)).flatten()
prefixes_repeated = np.repeat(census_v4['prefix'].values, 256)

census_ips = pd.DataFrame({
    'prefix': prefixes_repeated,
    'ip_int': all_ip_ints,
})

print(f"{len(census_ips):,} IPs from {census_v4['prefix'].nunique():,} prefixes")
census_ips.head()

3,672,576 IPs from 14,346 prefixes


,prefix,ip_int
0,1.0.0.0/24,16777216
1,1.0.0.0/24,16777217
2,1.0.0.0/24,16777218
3,1.0.0.0/24,16777219
4,1.0.0.0/24,16777220


In [6]:
import tarfile, zipfile, io, csv

# IPv4-mapped IPv6 address range: ::ffff:0.0.0.0 – ::ffff:255.255.255.255
IPV4_MAPPED_BASE = int(ipaddress.IPv6Address('::ffff:0.0.0.0'))    # 281470681743360
IPV4_MAPPED_MAX  = int(ipaddress.IPv6Address('::ffff:255.255.255.255'))

# Load DB23 (DB_23 = USAGETYPE), keeping only the IPv4-mapped range
records = []
with tarfile.open('ip2location_2026-02-16.tar.gz', 'r:gz') as tar:
    for member in tar.getmembers():
        if 'DB23' in member.name:
            f = tar.extractfile(member)
            with zipfile.ZipFile(io.BytesIO(f.read())) as zf:
                csv_name = 'IPV6-COUNTRY-REGION-CITY-LATITUDE-LONGITUDE-ISP-DOMAIN-MOBILE-USAGETYPE.CSV'
                with zf.open(csv_name) as csv_f:
                    reader = csv.reader(io.TextIOWrapper(csv_f, encoding='utf-8'))
                    for row in reader:
                        ip_from, ip_to = int(row[0]), int(row[1])
                        if ip_to >= IPV4_MAPPED_BASE and ip_from <= IPV4_MAPPED_MAX:
                            records.append((
                                max(ip_from, IPV4_MAPPED_BASE),
                                min(ip_to,   IPV4_MAPPED_MAX),
                                row[13],  # usage_type column
                            ))

db23_v4 = pd.DataFrame(records, columns=['ip_from', 'ip_to', 'usage_type'])
print(f"Loaded {len(db23_v4):,} IPv4 ranges from DB23")

# Vectorised range lookup via binary search
ip_int_mapped = census_ips['ip_int'].values + IPV4_MAPPED_BASE
ip_from_arr   = db23_v4['ip_from'].values
ip_to_arr     = db23_v4['ip_to'].values
usage_arr     = db23_v4['usage_type'].values

idx      = np.searchsorted(ip_from_arr, ip_int_mapped, side='right') - 1
safe_idx = np.maximum(idx, 0)
in_range = (idx >= 0) & (ip_int_mapped <= ip_to_arr[safe_idx])

census_ips['usage_type'] = np.where(in_range, usage_arr[safe_idx], 'Unknown')

# Table: usage_type -> number of IPs
usage_table = (
    census_ips
        .groupby('usage_type')
        .size()
        .reset_index(name='number_of_ips')
        .sort_values('number_of_ips', ascending=False)
)

usage_table

Loaded 12,021,239 IPv4 ranges from DB23


,usage_type,number_of_ips
2,DCH,2390139
0,CDN,1053944
5,ISP,94908
1,COM,78789
6,ISP/MOB,19453
11,SES,8451
3,EDU,8150
9,ORG,7224
4,GOV,5887
8,MOB,3072


In [15]:
total_ips = len(census_ips)

usage_table = (
    census_ips
    .groupby('usage_type')
    .size()
    .reset_index(name='number_of_ips')
    .sort_values('number_of_ips', ascending=False)
)

usage_table['pct'] = (usage_table['number_of_ips'] / total_ips * 100).round(0).astype(int)

# --- latex ---
rows = ""
for _, row in usage_table.iterrows():
    rows += f"        {row['usage_type']:<30} & \\num{{{row['number_of_ips']}}} & ({row['pct']}\\%) \\\\\n"

latex = f"""\\begin{{table}}[tb]
    \\centering
    \\begin{{tabular}}{{l r@{{~}}l}}
        \\toprule
        Usage Type & \\multicolumn{{2}}{{c}}{{IPs}} \\\\
        \\midrule
{rows}        \\midrule
        Total & \\num{{{total_ips}}} & (100\\%) \\\\
        \\bottomrule
    \\end{{tabular}}
    \\caption{{IP2Location DB23 usage type distribution of active anycast IPs (n=\\num{{{total_ips}}}).}}
    \\label{{tab:anycast_usage_type}}
\\end{{table}}"""

print(latex)

\begin{table}[tb]
    \centering
    \begin{tabular}{l r@{~}l}
        \toprule
        Usage Type & \multicolumn{2}{c}{IPs} \\
        \midrule
        DCH                            & \num{2390139} & (65\%) \\
        CDN                            & \num{1053944} & (29\%) \\
        ISP                            & \num{94908} & (3\%) \\
        COM                            & \num{78789} & (2\%) \\
        ISP/MOB                        & \num{19453} & (1\%) \\
        SES                            & \num{8451} & (0\%) \\
        EDU                            & \num{8150} & (0\%) \\
        ORG                            & \num{7224} & (0\%) \\
        GOV                            & \num{5887} & (0\%) \\
        MOB                            & \num{3072} & (0\%) \\
        MIL                            & \num{2303} & (0\%) \\
        RSV                            & \num{256} & (0\%) \\
        \midrule
        Total & \num{3672576} & (100\%) \\
        \bottomrule
    \end{

In [7]:
# Join IPs with ASN, expand MOAS entries
census_ips_asn = census_ips.merge(census_v4[['prefix', 'ASN']], on='prefix')
census_ips_asn = (
    census_ips_asn
        .assign(ASN=census_ips_asn['ASN'].str.split('_'))
        .explode('ASN')
)

# Unique (ASN, usage_type) pairs
asn_usage = census_ips_asn[['ASN', 'usage_type']].drop_duplicates()

# usage_type -> number of ASNs
usage_asn_table = (
    asn_usage
        .groupby('usage_type')['ASN']
        .nunique()
        .reset_index(name='number_of_asns')
        .sort_values('number_of_asns', ascending=False)
)
display(usage_asn_table)

# Distribution: how many ASNs have 1, 2, 3, ... distinct usage types
asn_type_counts = (
    asn_usage
        .groupby('ASN')['usage_type']
        .nunique()
        .value_counts()
        .sort_index()
        .rename_axis('number_of_usage_types')
        .reset_index(name='number_of_asns')
)
print("\nASNs by number of distinct usage types:")
display(asn_type_counts)

,usage_type,number_of_asns
2,DCH,778
5,ISP,145
1,COM,110
0,CDN,44
6,ISP/MOB,33
3,EDU,24
9,ORG,19
11,SES,17
4,GOV,11
8,MOB,10



ASNs by number of distinct usage types:


,number_of_usage_types,number_of_asns
0,1,992
1,2,61
2,3,12
3,4,6
4,5,3
5,7,1


In [8]:
# Unique (prefix, usage_type) pairs
prefix_usage = census_ips[['prefix', 'usage_type']].drop_duplicates()

# usage_type -> number of /24s
usage_prefix_table = (
    prefix_usage
        .groupby('usage_type')['prefix']
        .nunique()
        .reset_index(name='number_of_24s')
        .sort_values('number_of_24s', ascending=False)
)
display(usage_prefix_table)

# Distribution: how many prefixes have 1, 2, 3, ... distinct usage types
prefix_type_counts = (
    prefix_usage
        .groupby('prefix')['usage_type']
        .nunique()
        .value_counts()
        .sort_index()
        .rename_axis('number_of_usage_types')
        .reset_index(name='number_of_24s')
)
print("\n/24s by number of distinct usage types:")
display(prefix_type_counts)

,usage_type,number_of_24s
2,DCH,9361
0,CDN,4118
5,ISP,387
1,COM,352
6,ISP/MOB,78
3,EDU,38
11,SES,36
9,ORG,33
4,GOV,23
8,MOB,12



/24s by number of distinct usage types:


,number_of_usage_types,number_of_24s
0,1,14263
1,2,66
2,3,15
3,4,2


In [9]:
# ASes with multiple usage types: show their usage type sets
asn_multi = (
    asn_usage
        .groupby('ASN')['usage_type']
        .apply(set)
        .reset_index(name='usage_types')
)
asn_multi = asn_multi[asn_multi['usage_types'].apply(len) > 1].copy()
asn_multi['num_usage_types'] = asn_multi['usage_types'].apply(len)
asn_multi = asn_multi.sort_values('num_usage_types', ascending=False)

print(f"{len(asn_multi)} ASes with multiple usage types:")
display(asn_multi.head(20))

# Prefixes with multiple usage types: show their usage type sets
prefix_multi = (
    prefix_usage
        .groupby('prefix')['usage_type']
        .apply(set)
        .reset_index(name='usage_types')
)
prefix_multi = prefix_multi[prefix_multi['usage_types'].apply(len) > 1].copy()
prefix_multi['num_usage_types'] = prefix_multi['usage_types'].apply(len)
prefix_multi = prefix_multi.sort_values('num_usage_types', ascending=False)

print(f"\n{len(prefix_multi)} /24s with multiple usage types:")
display(prefix_multi.head(20))

83 ASes with multiple usage types:


,ASN,usage_types,num_usage_types
64,13335,"{CDN, SES, ISP, COM, ISP/MOB, EDU, DCH}",7
666,396982,"{ISP, COM, EDU, DCH, CDN}",5
775,42,"{GOV, MOB, ISP/MOB, EDU, DCH}",5
322,209242,"{GOV, MIL, EDU, DCH, CDN}",5
29,11878,"{EDU, ISP, DCH, COM}",4
915,54113,"{DCH, CDN, SES, COM}",4
1045,8220,"{EDU, ISP, DCH, COM}",4
157,16276,"{CDN, SES, DCH, COM}",4
158,16509,"{ISP, CDN, DCH, COM}",4
773,4181,"{EDU, ISP, ORG, COM}",4



83 /24s with multiple usage types:


,prefix,usage_types,num_usage_types
13770,80.80.9.0/24,"{EDU, ISP, DCH, COM}",4
13344,74.211.89.0/24,"{EDU, ISP, ORG, COM}",4
12725,64.47.2.0/24,"{ORG, DCH, COM}",3
6424,198.166.40.0/24,"{ISP, COM, ISP/MOB}",3
7155,207.200.208.0/24,"{ORG, DCH, COM}",3
7154,207.200.200.0/24,"{ORG, DCH, COM}",3
7672,23.234.82.0/24,"{EDU, DCH, COM}",3
7358,212.35.96.0/24,"{ISP, DCH, COM}",3
7671,23.234.81.0/24,"{ISP, DCH, COM}",3
7733,24.56.178.0/24,"{EDU, ISP, COM}",3
